In [ ]:
from google.cloud import bigquery
import os

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
client = bigquery.Client(project=PROJECT_ID, location="EU")

def run_query(title, sql):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    df = client.query(sql).to_dataframe()
    print(df.to_string(index=False))
    return df

: 

In [ ]:
# QUERY 1: Ingresos por mes 

q1 = """
SELECT
  FORMAT_TIMESTAMP('%Y-%m', o.order_date) AS mes,
  COUNT(DISTINCT o.order_id)              AS num_pedidos,
  ROUND(SUM(oi.unit_price * oi.quantity), 2) AS ingresos_totales
FROM `project-468c9077-a9c5-4e11-b14.tc_dataset.orders`      AS o
JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.order_items` AS oi
  ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY mes
ORDER BY mes
"""

df_q1 = run_query("Q1 — Ingresos por mes", q1)

In [ ]:
# QUERY 2: Top 10 productos más vendidos

q2 = """
SELECT
  p.product_name                             AS producto,
  p.brand                                    AS marca,
  SUM(oi.quantity)                           AS unidades_vendidas,
  ROUND(SUM(oi.unit_price * oi.quantity), 2) AS revenue_total
FROM `project-468c9077-a9c5-4e11-b14.tc_dataset.order_items` AS oi
JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.products`    AS p
  ON oi.product_id = p.product_id
GROUP BY producto, marca
ORDER BY unidades_vendidas DESC
LIMIT 10
"""

df_q2 = run_query("Q2 — Top 10 productos más vendidos", q2)

In [ ]:
# QUERY 3: Pedidos y ticket medio por país 
q3 = """
SELECT
  cu.country                     AS pais,
  COUNT(DISTINCT cu.customer_id) AS total_clientes,
  COUNT(DISTINCT o.order_id)     AS total_pedidos,
  ROUND(AVG(pay.amount), 2)      AS ticket_medio
FROM `project-468c9077-a9c5-4e11-b14.tc_dataset.customers` AS cu
LEFT JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.orders`   AS o
  ON cu.customer_id = o.customer_id
LEFT JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.payments` AS pay
  ON o.order_id = pay.order_id
GROUP BY pais
ORDER BY total_pedidos DESC
"""

df_q3 = run_query("Q3 — Pedidos y ticket medio por país", q3)

In [ ]:
# QUERY 4: Tiempo medio de entrega por país
q4 = """
SELECT
  s.shipping_country AS pais,
  ROUND(AVG(
    TIMESTAMP_DIFF(s.delivery_date, o.order_date, DAY)
  ), 1)              AS dias_entrega_medio,
  COUNT(*)           AS pedidos_entregados
FROM `project-468c9077-a9c5-4e11-b14.tc_dataset.shipments` AS s
JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.orders`    AS o
  ON s.order_id = o.order_id
WHERE s.shipping_status = 'delivered'
GROUP BY pais
ORDER BY dias_entrega_medio ASC
"""

df_q4 = run_query("Q4 — Tiempo medio de entrega por país", q4)

In [ ]:
# QUERY 5: Margen de beneficio por categoría 

q5 = """
SELECT
  cat.name                                                    AS categoria,
  ROUND(SUM(oi.unit_price * oi.quantity), 2)                  AS ingresos,
  ROUND(SUM(p.cost_price  * oi.quantity), 2)                  AS coste,
  ROUND(SUM((oi.unit_price - p.cost_price) * oi.quantity), 2) AS margen_bruto,
  ROUND(
    100.0 * SUM((oi.unit_price - p.cost_price) * oi.quantity)
    / NULLIF(SUM(oi.unit_price * oi.quantity), 0)
  , 2)                                                        AS margen_pct
FROM `project-468c9077-a9c5-4e11-b14.tc_dataset.order_items` AS oi
JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.products`    AS p
  ON oi.product_id = p.product_id
JOIN `project-468c9077-a9c5-4e11-b14.tc_dataset.categories`  AS cat
  ON p.category_id = cat.category_id
GROUP BY categoria
ORDER BY margen_pct DESC
"""

df_q5 = run_query("Q5 — Margen de beneficio por categoría", q5)